Cell 1 — Mount Drive

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


Cell 2 — Install packages

In [2]:
!pip install -q transformers sentencepiece fair-esm captum goatools biopython \
    pyyaml tqdm joblib scipy seaborn matplotlib pandas numpy scikit-learn

# CD-HIT creates the homology-aware split.
!apt-get -qq update
!apt-get -qq install -y cd-hit

print("Packages and CD-HIT are ready.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 60.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 13.2 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package cd-hit.
(Reading database ... 118332 files and directories currently installed.)
Preparing to unpack .../cd-hit_4.8.1-4_amd64.deb ...
Unpacking cd-hit (4.8.1-4) ...
Setting up cd-hit (4.8.1-4) ...
Processing triggers for man-db (2.10.2-1) ...
Packages and CD-HIT are ready.


Cell 3 — Create project folders

In [3]:
import os

PROJECT_ROOT = "/content/drive/MyDrive/atlas-go-revision"

folders = [
    f"{PROJECT_ROOT}/data/raw",
    f"{PROJECT_ROOT}/data/processed",
    f"{PROJECT_ROOT}/data/processed/models",
    f"{PROJECT_ROOT}/results/tables",
    f"{PROJECT_ROOT}/results/figures",
    f"{PROJECT_ROOT}/configs",
    f"{PROJECT_ROOT}/src",
    f"{PROJECT_ROOT}/submission_files",
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Project folder:")
print(PROJECT_ROOT)

Project folder:
/content/drive/MyDrive/atlas-go-revision


Cell 4 — Check raw files

In [4]:
raw_folder = f"{PROJECT_ROOT}/data/raw"

required_files = [
    "ATLAS.csv",
    "sequence_GoLabels.csv",
    "go-basic.obo",
]

for filename in required_files:
    filepath = f"{raw_folder}/{filename}"

    if os.path.exists(filepath):
        size_mb = os.path.getsize(filepath) / (1024 * 1024)
        print(f"✓ {filename}: {size_mb:.2f} MB")
    else:
        print(f"✗ Missing: {filename}")

✓ ATLAS.csv: 0.23 MB
✓ sequence_GoLabels.csv: 1.30 MB
✓ go-basic.obo: 29.96 MB


Cell 5 — Create config.yaml

In [5]:
import yaml

config = {
    "paths": {
        "project_root": PROJECT_ROOT,
        "data_raw": f"{PROJECT_ROOT}/data/raw",
        "data_processed": f"{PROJECT_ROOT}/data/processed",
        "results": f"{PROJECT_ROOT}/results/tables",
        "figures": f"{PROJECT_ROOT}/results/figures",
        "models": f"{PROJECT_ROOT}/data/processed/models",
    },

    "data": {
        # Keep GO terms occurring in at least five proteins.
        "min_proteins_per_go": 5,

        # Important: no minimum GO annotations per protein is applied.
        # Proteins are removed only if they have zero retained labels
        # AFTER rare GO terms have been removed.
        "remove_zero_label_proteins": True,

        "train_ratio": 0.80,
        "val_ratio": 0.10,
        "test_ratio": 0.10,

        # Same sequence coverage for both protein language models.
        "protbert_max_residues": 1000,
        "esm2_max_residues": 1000,

        "static_features": ["α%", "β%", "Coil%", "Len."],
        "dynamic_features": ["Avg. RMSF", "Avg. gyr.", "Div. SE", "Div. MM"],
    },

    # Main revised evaluation design: CD-HIT homology-aware split.
    "split": {
        "main_split": "homology",
        "random_seed_for_split": 42,
    },

    "homology_split": {
        "enabled": True,
        "method": "cdhit",
        "identity_threshold": 0.40,
        "word_size": 2,
        "cdhit_binary": "cd-hit",
    },

    # GO true-path rule: annotations and prediction scores are propagated
    # upward to GO ancestors before CAFA-style evaluation.
    "go_hierarchy": {
        "enabled": True,
        "obo_path": f"{PROJECT_ROOT}/data/raw/go-basic.obo",
        "propagate_labels": True,
        "propagate_predictions": True,
    },

    # Reviewer-requested repeated training.
    "training_seeds": [42, 1, 7],

    "training": {
        "max_epochs": 100,
        "early_stopping_patience": 10,
        "gradient_clip_norm": 1.0,
        "focal_loss_alpha": 0.75,
        "focal_loss_gamma": 2.0,
        "cosine_annealing_tmax": 50,

        "mlp": {
            "batch_size": 256,
            "learning_rate": 3e-4,
            "weight_decay": 1e-4,
        },

        "cnn": {
            "batch_size": 256,
            "learning_rate": 1e-4,
            "weight_decay": 1e-5,
        },

        # Correct name: residual feed-forward network, not TransformerGO.
        "resffn": {
            "batch_size": 128,
            "learning_rate": 1e-4,
            "weight_decay": 1e-5,
            "hidden_dim": 512,
            "num_layers": 3,
            "dropout": 0.1,
        },

    },

    # Hyperparameters must be selected using validation Fmax only.
    "hyperparameter_search": {
        "enabled": True,
        "feature_set_for_tuning": "pes",
        "learning_rates": [1e-4, 2e-4, 3e-4],
        "weight_decays": [1e-5, 1e-4],
    },

    # Reviewer-requested standard frequency baseline.
    "baseline": {
        "name": "continuous_training_frequency",
        "evaluate_all_go_terms": True,
    },

    # Confidence intervals for key results.
    "bootstrap": {
        "enabled": True,
        "n_resamples": 1000,
        "confidence_level": 0.95,
        "random_seed": 0,
    },

    "feature_extraction": {
        # Conservative settings for a free Colab T4 GPU.
        "protbert_batch_size": 16,
        "esm2_batch_size": 4,
    },

    "xai": {
        "enabled": True,
        "integrated_gradients_steps": 50,
        "permutation_repeats": 20,

        # Required reviewer confirmation method.
        "run_leave_one_modality_out": True,

        # Do not analyze only the highest-scoring GO term.
        "terms_per_protein": {
            "top1": 1,
            "correct_terms": 2,
            "incorrect_terms": 2,
            "rare_terms": 1,
        },
    },
}

config_path = f"{PROJECT_ROOT}/configs/config.yaml"

with open(config_path, "w") as file:
    yaml.dump(config, file, sort_keys=False)

print("Saved configuration:")
print(config_path)

Saved configuration:
/content/drive/MyDrive/atlas-go-revision/configs/config.yaml


Cell 6 — Save environment requirements for reproducibility

In [6]:
requirements = """
transformers
sentencepiece
fair-esm
captum
goatools
biopython
PyYAML
tqdm
joblib
scipy
seaborn
matplotlib
pandas
numpy
scikit-learn
torch
"""

requirements_path = f"{PROJECT_ROOT}/requirements.txt"

with open(requirements_path, "w") as file:
    file.write(requirements.strip())

print("Saved:", requirements_path)

!pip freeze > "$PROJECT_ROOT/package_versions.txt"

print("Saved package_versions.txt")

Saved: /content/drive/MyDrive/atlas-go-revision/requirements.txt
Saved package_versions.txt


Cell 7 — Verify the design

In [7]:
with open(config_path, "r") as file:
    cfg = yaml.safe_load(file)

n_features = 9
n_architectures = 3
n_seeds = len(cfg["training_seeds"])

print("=" * 60)
print("REVISED PIPELINE SETUP COMPLETE")
print("=" * 60)
print("Main split:", cfg["split"]["main_split"])
print("CD-HIT identity threshold:", cfg["homology_split"]["identity_threshold"])
print("GO hierarchy enabled:", cfg["go_hierarchy"]["enabled"])
print("Rare GO-term threshold:", cfg["data"]["min_proteins_per_go"])
print("Minimum annotations per protein: NOT USED")
print("ProtBERT maximum residues:", cfg["data"]["protbert_max_residues"])
print("ESM2 maximum residues:", cfg["data"]["esm2_max_residues"])
print("Training seeds:", cfg["training_seeds"])
print()
print("Training runs:")
print(f"{n_features} feature sets × {n_architectures} architectures × {n_seeds} seeds")
print(f"= {n_features * n_architectures * n_seeds} total runs")
print()
print("Next notebook: 01_data_preparation.ipynb")

REVISED PIPELINE SETUP COMPLETE
Main split: homology
CD-HIT identity threshold: 0.4
GO hierarchy enabled: True
Rare GO-term threshold: 5
Minimum annotations per protein: NOT USED
ProtBERT maximum residues: 1000
ESM2 maximum residues: 1000
Training seeds: [42, 1, 7]

Training runs:
9 feature sets × 3 architectures × 3 seeds
= 81 total runs

Next notebook: 01_data_preparation.ipynb
